# Apache Spark and PySpark I, Architecture, DataFrames, Joins and Windows

Notebook for **Chapter 9** of the *de* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/de/ch09.html

Tip: change a number, predict the output, then run the cell again.

## 9.1 The Architecture: Driver, Executors, Jobs, Stages, Tasks

## 9.2 DataFrames, Transformations vs Actions, and Lazy Evaluation

In [ ]:
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.appName("shopkart").getOrCreate()

orders = spark.read.parquet("s3://shopkart-lake/silver/orders/")   # transformation: lazy
big = (orders
       .filter(F.col("amount") >= 100)                 # narrow
       .withColumn("amount_f", F.col("amount").cast("double")))   # narrow
print(big.count())        # ACTION: the job runs now
big.explain()             # prints the physical plan; no job

In [ ]:
(orders.write.mode("overwrite")
       .partitionBy("order_date")
       .parquet("s3://shopkart-lake/gold/orders/"))      # not run here

In [ ]:
# not run here
@F.udf("double")
def rupees(paise): return paise / 100.0          # slow: use F.col("paise") / 100 instead

## 9.3 Joins and Window Functions on Partitioned Data

In [ ]:
joined  = orders.join(customers, on="customer_id", how="inner")
missing = orders.join(customers, on="customer_id", how="left_anti")   # orphan orders

In [ ]:
from pyspark.sql import Window
w_day  = Window.partitionBy("customer_id").orderBy("day")
w_amt  = Window.partitionBy("customer_id").orderBy(F.col("amount").desc())
out = (orders
       .withColumn("running", F.sum("amount").over(w_day))
       .withColumn("rn", F.row_number().over(w_amt)))

## 9.4 More Worked Examples: Counting Tasks, Python Cost and Group-wise Pandas

Three more problems build the habit that Spark experts have: before you run a job, predict how many tasks it creates, where Python code is slow, and how much memory one group needs. Each example runs here with plain Python or pandas. We say clearly which behaviour comes from Spark and cannot be run here.

In [ ]:
# How many tasks does a read create? Splittable files are cut at maxPartitionBytes (128 MB default);
# a gzip-compressed CSV cannot be split, so one file is one task whatever its size.
import math
MB = 1
def tasks(files, splittable, max_part=128):
    if not splittable: return len(files)
    return sum(math.ceil(f / max_part) for f in files)
cases = {
  "40 Parquet files of 250 MB": ([250] * 40, True),
  "1 Parquet file of 10 GB": ([10_240], True),
  "1 gzip CSV of 10 GB": ([10_240], False),
  "2,000 Parquet files of 2 MB": ([2] * 2000, True),
}
cores = 40
for name, (files, split) in cases.items():
    n = tasks(files, split)
    waves = math.ceil(n / cores)
    print(f"{name:30s} tasks={n:5d} waves on {cores} cores={waves}")
print("note: real Spark also packs small files together up to the partition size, so the 2,000 files case is smaller in practice")

40 Parquet files of 250 MB     tasks=   80 waves on 40 cores=2
1 Parquet file of 10 GB        tasks=   80 waves on 40 cores=2
1 gzip CSV of 10 GB            tasks=    1 waves on 40 cores=1
2,000 Parquet files of 2 MB    tasks= 2000 waves on 40 cores=50
note: real Spark also packs small files together up to the partition size, so the 2,000 files case is smaller in practice


In [ ]:
# Why a Python UDF is slow and a pandas (Arrow) UDF is faster: count the calls between the JVM and Python.
import math
rows = 1_000_000
batch = 10_000            # spark.sql.execution.arrow.maxRecordsPerBatch default
print("row-at-a-time Python UDF: Python function calls =", rows)
print("pandas UDF: Python function calls =", math.ceil(rows / batch), "(one per batch of", batch, "rows)")
print("ratio:", rows // math.ceil(rows / batch), "times fewer calls")
# the logic itself, run here with plain pandas, is what a pandas UDF would do on each batch
import pandas as pd
s = pd.Series([100.0, 250.0, 80.0])
print("vectorized tax:", (s * 0.18).round(2).tolist())

row-at-a-time Python UDF: Python function calls = 1000000
pandas UDF: Python function calls = 100 (one per batch of 10000 rows)
ratio: 10000 times fewer calls
vectorized tax: [18.0, 45.0, 14.4]


In [ ]:
# applyInPandas semantics, shown with plain pandas: Spark calls your function once per group, on an executor.
import pandas as pd
df = pd.DataFrame({"user": ["a", "a", "a", "b", "b"], "ts": [1, 2, 3, 1, 2], "amount": [10.0, 30.0, 20.0, 5.0, 7.0]})
def add_running_total(g):
    g = g.sort_values("ts").copy()
    g["running"] = g["amount"].cumsum()
    return g
out = pd.concat([add_running_total(g) for _, g in df.groupby("user")])
print(out.to_string(index=False))
print("groups processed:", df["user"].nunique(), "| largest group must fit in one executor's memory:", df.groupby("user").size().max(), "rows")

user  ts  amount  running
   a   1    10.0     10.0
   a   2    30.0     40.0
   a   3    20.0     60.0
   b   1     5.0      5.0
   b   2     7.0     12.0
groups processed: 2 | largest group must fit in one executor's memory: 3 rows


## Chapter recap

## Practice set

**1.** (easy) Is `filter` a transformation or an action? **Answer:** a (narrow) transformation.

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/de/ch09.html